# 🌐 Notebook 4: Cross-Dataset Generalization Evaluation
This 100% self-contained evaluation notebook directly answers the CRITIC question: **"Is it better to train on Crack500 and test on DeepCrack? How do we solve cross-dataset overfitting?"**

It evaluates all baseline and KD models trained on **Crack500** directly on the **DeepCrack** test split (and vice versa) without fine-tuning on the target domain, quantifying domain transfer gap and KD generalization gains.


In [1]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box data/teacher_logits_centroid runs


In [2]:
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.3 MB/s eta 0:00:00


In [3]:
%%writefile scripts/convert_crack500.py
#!/usr/bin/env python3
"""
Crack500 → YOLO seg format converter
=====================================
Crack500 structure:
  crack500/
  ├── traincrop/   ← 00001.jpg + 00001.png (binary mask, same stem)
  ├── valcrop/
  ├── testcrop/
  ├── train.txt    ← list of image filenames (optional)
  ├── val.txt
  └── test.txt

Output (YOLO seg format, ready for ultralytics):
  crack500_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Usage:
  python scripts/convert_crack500.py \
      --src ~/distill/data/datasets/crack500 \
      --dst ~/distill/data/datasets/crack500_yolo
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.

    Returns list of label lines (one per instance).
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (Crack500 masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_split(src_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process one split (train/val/test)."""

    # Crack500 stores images+masks together in traincrop/valcrop/testcrop
    crop_dir = src_dir / f"{split_name}crop"
    if not crop_dir.exists():
        # Try alternate names
        for candidate in [src_dir / split_name, src_dir / f"{split_name}data"]:
            if candidate.exists():
                crop_dir = candidate
                break
        else:
            print(f"  [WARNING] Could not find directory for split '{split_name}', skipping.")
            return 0

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    # Find all images (jpg/jpeg/png that are NOT masks)
    all_files = sorted(crop_dir.iterdir())
    # Crack500: image = .jpg, mask = same stem + .png
    image_files = [f for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')
                   and ':Zone.Identifier' not in f.name]

    if not image_files:
        # Some versions store as .png images too — distinguish by paired files
        png_files = [f for f in all_files if f.suffix.lower() == '.png'
                     and ':Zone.Identifier' not in f.name]
        # If .jpg exists for a stem → .png is mask. If no .jpg → .png is image.
        jpg_stems = {f.stem for f in all_files if f.suffix.lower() in ('.jpg', '.jpeg')}
        image_files = [f for f in png_files if f.stem not in jpg_stems]

    converted = 0
    skipped   = 0

    for img_path in tqdm(image_files, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem)
        mask_path = crop_dir / f"{stem}.png"
        if not mask_path.exists():
            # Try .bmp
            mask_path = crop_dir / f"{stem}.bmp"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# Crack500 — YOLO seg format
# Auto-generated by convert_crack500.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert Crack500 to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        required=True,
        help="Path to crack500 root dir (contains traincrop/, valcrop/, testcrop/)"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    counts = {}
    for split in ["train", "val", "test"]:
        n = process_split(
            src_dir    = src,
            dst_img_dir= dst / "images" / split,
            dst_lbl_dir= dst / "labels" / split,
            split_name = split,
        )
        counts[split] = n

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted dataset at: {dst}")
    print(f"\nNext step — test YOLO11 loads it:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('yolo11n-seg.pt')")
    print(f"  model.train(data='{dst}/dataset.yaml', epochs=1, imgsz=512)")


if __name__ == "__main__":
    main()
# (appended — nothing, file is complete)


Writing scripts/convert_crack500.py


In [4]:
%%writefile scripts/convert_deepcrack.py
#!/usr/bin/env python3
"""
DeepCrack → YOLO seg format converter
=====================================
DeepCrack structure:
  deepcrack/
  ├── train_img/      ← 11111.jpg
  ├── train_lab/      ← 11111.png (binary mask, 0/255)
  ├── test_img/       ← 111212-1.jpg
  └── test_lab/       ← 111212-1.png (binary mask, 0/255)

Output (YOLO seg format, ready for ultralytics):
  deepcrack_yolo/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  ├── labels/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── dataset.yaml

Each .txt label: one line per connected crack instance
  0 x1 y1 x2 y2 ... (normalized polygon, class 0 = crack)

Splits:
  - Train: 80% of train_img (480 images)
  - Val: 20% of train_img (120 images)
  - Test: 100% of test_img (474 images)
"""

import os
import cv2
import numpy as np
import argparse
import shutil
from pathlib import Path
from tqdm import tqdm


CLASS_ID = 0        # single class: crack
MIN_AREA = 50       # minimum pixel area to keep an instance
MIN_POINTS = 6      # minimum polygon points (3 coordinate pairs)


def binary_mask_to_yolo_instances(mask_path: str, img_w: int, img_h: int) -> list[str]:
    """
    Read binary PNG mask → split into instances via connectedComponents
    → convert each to normalized YOLO seg polygon string.
    """
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    # Threshold (DeepCrack masks are 0/255)
    binary = (mask > 127).astype(np.uint8)

    # Separate touching cracks into individual instances
    num_labels, labels_map = cv2.connectedComponents(binary)

    label_lines = []
    for label_id in range(1, num_labels):      # 0 = background
        instance = (labels_map == label_id).astype(np.uint8)

        if instance.sum() < MIN_AREA:
            continue

        # Find contours for this instance
        contours, _ = cv2.findContours(
            instance, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:
            if len(contour) < MIN_POINTS // 2:
                continue

            # Flatten and normalize to [0, 1]
            pts = contour.squeeze()
            if pts.ndim == 1:
                pts = pts.reshape(1, 2)

            # Simplify contour slightly to reduce file size
            epsilon = 0.002 * cv2.arcLength(contour, True)
            simplified = cv2.approxPolyDP(contour, epsilon, True).squeeze()
            if simplified.ndim == 1:
                simplified = simplified.reshape(1, 2)
            if len(simplified) < 3:
                simplified = pts

            norm = []
            for x, y in simplified:
                norm.append(x / img_w)
                norm.append(y / img_h)

            if len(norm) < MIN_POINTS:
                continue

            coords_str = " ".join(f"{v:.6f}" for v in norm)
            label_lines.append(f"{CLASS_ID} {coords_str}")

    return label_lines


def process_images(image_list: list[Path], mask_dir: Path, dst_img_dir: Path, dst_lbl_dir: Path, split_name: str):
    """Process a list of images for a specific split."""
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    converted = 0
    skipped = 0

    for img_path in tqdm(image_list, desc=f"  {split_name}", leave=False):
        stem = img_path.stem

        # Find corresponding mask (.png with same stem in mask_dir)
        mask_path = mask_dir / f"{stem}.png"
        if not mask_path.exists():
            skipped += 1
            continue

        # Read image to get dimensions
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        h, w = img.shape[:2]

        # Convert mask to YOLO seg labels
        label_lines = binary_mask_to_yolo_instances(str(mask_path), w, h)

        # Copy image
        dst_img_path = dst_img_dir / img_path.name
        shutil.copy2(img_path, dst_img_path)

        # Write label file (even if empty — YOLO needs it)
        dst_lbl_path = dst_lbl_dir / f"{stem}.txt"
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        converted += 1

    print(f"  {split_name}: {converted} images converted, {skipped} skipped")
    return converted


def write_dataset_yaml(dst: Path, num_train: int, num_val: int, num_test: int):
    """Write ultralytics-compatible dataset.yaml."""
    yaml_content = f"""# DeepCrack — YOLO seg format
# Auto-generated by convert_deepcrack.py

path: {dst.resolve()}
train: images/train
val:   images/val
test:  images/test

nc: 1
names:
  0: crack

# Stats
# train: ~{num_train} images
# val:   ~{num_val} images
# test:  ~{num_test} images
"""
    with open(dst / "dataset.yaml", "w") as f:
        f.write(yaml_content)
    print(f"\n  dataset.yaml written to {dst / 'dataset.yaml'}")


def verify_conversion(dst: Path):
    """Quick sanity check on converted dataset."""
    print("\n[Verify] Checking converted dataset...")
    issues = 0
    for split in ["train", "val", "test"]:
        img_dir = dst / "images" / split
        lbl_dir = dst / "labels" / split
        if not img_dir.exists():
            continue

        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
        lbls = list(lbl_dir.glob("*.txt"))

        # Check counts match
        if len(imgs) != len(lbls):
            print(f"  [!] {split}: {len(imgs)} images vs {len(lbls)} labels — mismatch!")
            issues += 1
        else:
            print(f"  {split}: {len(imgs)} images ✓")

        # Check a few labels are non-empty
        non_empty = sum(1 for l in lbls if l.stat().st_size > 0)
        empty     = len(lbls) - non_empty
        print(f"    labels with cracks: {non_empty} | empty (no crack): {empty}")

        if non_empty == 0:
            print(f"  [!] {split}: ALL labels are empty — check mask paths!")
            issues += 1

    if issues == 0:
        print("\n  ✓ Dataset looks good!")
    else:
        print(f"\n  ✗ {issues} issue(s) found — check output above.")

    return issues == 0


def main():
    parser = argparse.ArgumentParser(description="Convert DeepCrack to YOLO seg format")
    parser.add_argument(
        "--src",
        type=str,
        default="data/datasets/deepcrack",
        help="Path to deepcrack root dir"
    )
    parser.add_argument(
        "--dst",
        type=str,
        default=None,
        help="Output directory (default: <src>_yolo)"
    )
    parser.add_argument(
        "--verify",
        action="store_true",
        default=True,
        help="Run sanity check after conversion"
    )
    args = parser.parse_args()

    src = Path(args.src).expanduser().resolve()
    dst = Path(args.dst).expanduser().resolve() if args.dst else src.parent / f"{src.name}_yolo"

    print(f"[Convert] Source: {src}")
    print(f"[Convert] Output: {dst}")
    print()

    if not src.exists():
        print(f"ERROR: Source directory not found: {src}")
        return

    # Check and clean output directory if exists
    if dst.exists():
        print(f"[Warning] Output directory exists, clearing: {dst}")
        shutil.rmtree(dst)

    # 1. Process Train split and do 80-20 partition
    train_img_dir = src / "train_img"
    train_lab_dir = src / "train_lab"
    
    if not train_img_dir.exists() or not train_lab_dir.exists():
        print(f"ERROR: Train directories not found under {src}")
        return

    all_train_files = sorted([
        f for f in train_img_dir.iterdir()
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and ':Zone.Identifier' not in f.name
    ])

    # Deterministic shuffle
    rng = np.random.RandomState(42)
    shuffled_indices = rng.permutation(len(all_train_files))
    
    n_train = int(len(all_train_files) * 0.8)
    
    train_files = [all_train_files[i] for i in shuffled_indices[:n_train]]
    val_files   = [all_train_files[i] for i in shuffled_indices[n_train:]]

    print(f"Total training images: {len(all_train_files)}")
    print(f"  -> Train split: {len(train_files)}")
    print(f"  -> Val split:   {len(val_files)}")

    # 2. Process Test split
    test_img_dir = src / "test_img"
    test_lab_dir = src / "test_lab"
    
    if not test_img_dir.exists() or not test_lab_dir.exists():
        print(f"ERROR: Test directories not found under {src}")
        return

    test_files = sorted([
        f for f in test_img_dir.iterdir()
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        and ':Zone.Identifier' not in f.name
    ])
    print(f"Total test images:     {len(test_files)}")

    # Convert splits
    counts = {}
    counts["train"] = process_images(
        image_list = train_files,
        mask_dir   = train_lab_dir,
        dst_img_dir= dst / "images" / "train",
        dst_lbl_dir= dst / "labels" / "train",
        split_name = "train",
    )
    
    counts["val"] = process_images(
        image_list = val_files,
        mask_dir   = train_lab_dir,
        dst_img_dir= dst / "images" / "val",
        dst_lbl_dir= dst / "labels" / "val",
        split_name = "val",
    )

    counts["test"] = process_images(
        image_list = test_files,
        mask_dir   = test_lab_dir,
        dst_img_dir= dst / "images" / "test",
        dst_lbl_dir= dst / "labels" / "test",
        split_name = "test",
    )

    write_dataset_yaml(dst, counts["train"], counts["val"], counts["test"])

    if args.verify:
        verify_conversion(dst)

    print(f"\n[Done] Converted DeepCrack dataset at: {dst}")


if __name__ == "__main__":
    main()


Writing scripts/convert_deepcrack.py


In [5]:
import os
import shutil
from pathlib import Path

# Priority check for custom dataset input folder
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists():
    input_dir = Path("/kaggle/input")

print(f"Scanning input directory: {input_dir}")
if input_dir.exists():
    try:
        print("Direct contents:", os.listdir(str(input_dir)))
    except Exception as e:
        print("Error listing input directory:", e)

datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = Path("checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# 1. Clean up any existing local dataset directories/symlinks to avoid read-only collisions
for folder in ["combined_yolo", "crack500_yolo", "crack500_uncropped_yolo", "deepcrack_yolo", "crack500", "deepcrack"]:
    p = datasets_dir / folder
    if os.path.lexists(p):
        if os.path.islink(p): os.unlink(p)
        else: shutil.rmtree(p)

# 2. Clean up teacher logits folders and link them to /tmp to redirect disk usage
for folder in ["teacher_logits_box", "teacher_logits_centroid", "teacher_features"]:
    p_local = Path("data") / folder
    p_tmp = Path("/tmp") / folder
    if os.path.lexists(p_local):
        if os.path.islink(p_local): os.unlink(p_local)
        else: shutil.rmtree(p_local)
    p_tmp.mkdir(parents=True, exist_ok=True)
    os.symlink(p_tmp, p_local)
    print(f"Redirected {p_local} -> {p_tmp}")

# 3. Locate and link SAM 2 weights
linked_sam = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "sam2_hiera_large.pt" in files and not linked_sam:
        dest = checkpoints_dir / "sam2_hiera_large.pt"
        if os.path.lexists(dest): os.remove(dest)
        os.symlink(root_path / "sam2_hiera_large.pt", dest)
        print(f"Linked SAM 2 checkpoint: {root_path / 'sam2_hiera_large.pt'} -> {dest}")
        linked_sam = True

if not (checkpoints_dir / "sam2_hiera_large.pt").exists():
    print("SAM 2 checkpoint not found in inputs. Downloading...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt -O checkpoints/sam2_hiera_large.pt

# 4. Link raw datasets for conversion
linked_crack = False
linked_deep = False
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs and not linked_crack:
        dest = datasets_dir / "crack500"
        os.symlink(root_path, dest)
        print(f"Linked raw Crack500: {root_path} -> {dest}")
        linked_crack = True
    if "train_img" in dirs and not linked_deep:
        dest = datasets_dir / "deepcrack"
        os.symlink(root_path, dest)
        print(f"Linked raw DeepCrack: {root_path} -> {dest}")
        linked_deep = True

if not (datasets_dir / "crack500").exists():
    print("WARNING: Raw Crack500 dataset not linked!")
if not (datasets_dir / "deepcrack").exists():
    print("WARNING: Raw DeepCrack dataset not linked!")

# 5. Scan and link trained model checkpoints from /kaggle/input
print("\nScanning for trained model checkpoints in input directory...")
runs_dir = Path("runs")
runs_dir.mkdir(parents=True, exist_ok=True)

ckpt_target_map = {
    "v8_crack500_baseline": "runs/baselines/v8_crack500_baseline/weights/best.pt",
    "v11_crack500_baseline": "runs/baselines/v11_crack500_baseline/weights/best.pt",
    "v11_crack500_full_kd": "runs/crack_distill_full_kd_box_instance_seg_yolo11n-seg/weights/best.pt",
    "v8_deepcrack_baseline": "runs/baselines/v8_deepcrack_baseline/weights/best.pt",
    "v11_deepcrack_baseline": "runs/baselines/v11_deepcrack_baseline/weights/best.pt",
    "v11_deepcrack_full_kd": "runs/crack_distill_full_kd_box_instance_seg_deepcrack/weights/best.pt",
}

found_ckpts = {}
for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    for f in sorted(files):
        if f in ["best.pt", "last.pt"]:
            full_p = root_path / f
            full_p_str = str(full_p).lower()

            # Flexible matching rules for each experiment key
            matched_key = None
            if "v8_crack500_baseline" in full_p_str or ("nb1" in full_p_str and "v8" in full_p_str and "crack500" in full_p_str):
                matched_key = "v8_crack500_baseline"
            elif "v11_crack500_baseline" in full_p_str or ("nb1" in full_p_str and "v11" in full_p_str and "crack500" in full_p_str):
                matched_key = "v11_crack500_baseline"
            elif "v8_deepcrack_baseline" in full_p_str or ("nb1" in full_p_str and "v8" in full_p_str and "deepcrack" in full_p_str):
                matched_key = "v8_deepcrack_baseline"
            elif "v11_deepcrack_baseline" in full_p_str or ("nb1" in full_p_str and "v11" in full_p_str and "deepcrack" in full_p_str):
                matched_key = "v11_deepcrack_baseline"
            elif "v11_crack500_full_kd" in full_p_str or ("nb2" in full_p_str and ("kd" in full_p_str or "full_kd" in full_p_str)) or ("crack500" in full_p_str and "full_kd" in full_p_str):
                matched_key = "v11_crack500_full_kd"
            elif "v11_deepcrack_full_kd" in full_p_str or ("nb3" in full_p_str and ("kd" in full_p_str or "full_kd" in full_p_str or "seghead" in full_p_str)) or ("deepcrack" in full_p_str and ("full_kd" in full_p_str or "seghead_frozen" in full_p_str)):
                matched_key = "v11_deepcrack_full_kd"

            if matched_key:
                # Prefer best.pt over last.pt if already found
                if matched_key in found_ckpts and f == "last.pt" and "best.pt" in str(found_ckpts[matched_key]):
                    continue
                target_rel_path = ckpt_target_map[matched_key]
                target_path = Path(target_rel_path)
                target_path.parent.mkdir(parents=True, exist_ok=True)
                if os.path.lexists(target_path):
                    if os.path.islink(target_path): os.unlink(target_path)
                    else: os.remove(target_path)
                os.symlink(full_p, target_path)
                found_ckpts[matched_key] = full_p
                print(f"  [Found Checkpoint] {matched_key}: {full_p} -> {target_path}")

print(f"Total model checkpoints linked: {len(found_ckpts)}/6")


Scanning input directory: /kaggle/input
Direct contents: ['notebooks', 'datasets']
Redirected data/teacher_logits_box -> /tmp/teacher_logits_box
Redirected data/teacher_logits_centroid -> /tmp/teacher_logits_centroid
Redirected data/teacher_features -> /tmp/teacher_features
Linked SAM 2 checkpoint: /kaggle/input/notebooks/rauffatali/nb1-baseline/checkpoints/sam2_hiera_large.pt -> checkpoints/sam2_hiera_large.pt
Linked raw DeepCrack: /kaggle/input/datasets/rauffatali/distill-datasetforme/deepcrack -> data/datasets/deepcrack
Linked raw Crack500: /kaggle/input/datasets/rauffatali/distill-datasetforme/crack500 -> data/datasets/crack500

Scanning for trained model checkpoints in input directory...
  [Found Checkpoint] v11_deepcrack_baseline: /kaggle/input/notebooks/rauffatali/nb1-baseline/runs/segment/runs/baselines/v11_deepcrack_baseline/weights/best.pt -> runs/baselines/v11_deepcrack_baseline/weights/best.pt
  [Found Checkpoint] v11_crack500_baseline: /kaggle/input/notebooks/rauffatali/nb

In [6]:
# Convert datasets to YOLO format for test evaluation
!python scripts/convert_crack500.py --src data/datasets/crack500 --dst data/datasets/crack500_yolo
!python scripts/convert_deepcrack.py --src data/datasets/deepcrack --dst data/datasets/deepcrack_yolo


[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/crack500
[Convert] Output: /kaggle/working/data/datasets/crack500_yolo

  train: 1896 images converted, 0 skipped
  val: 348 images converted, 0 skipped
  test: 1124 images converted, 0 skipped

  dataset.yaml written to /kaggle/working/data/datasets/crack500_yolo/dataset.yaml

[Verify] Checking converted dataset...
  train: 1896 images ✓
    labels with cracks: 1896 | empty (no crack): 0
  val: 348 images ✓
    labels with cracks: 348 | empty (no crack): 0
  test: 1124 images ✓
    labels with cracks: 1124 | empty (no crack): 0

  ✓ Dataset looks good!

[Done] Converted dataset at: /kaggle/working/data/datasets/crack500_yolo

Next step — test YOLO11 loads it:
  from ultralytics import YOLO
  model = YOLO('yolo11n-seg.pt')
  model.train(data='/kaggle/working/data/datasets/crack500_yolo/dataset.yaml', epochs=1, imgsz=512)
[Convert] Source: /kaggle/input/datasets/rauffatali/distill-datasetforme/deepcrack
[Convert] O

In [7]:
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

# Define models to evaluate cross-dataset
eval_matrix = [
    # Train: Crack500 -> Test: DeepCrack
    {"name": "v8_crack500_baseline", "train": "Crack500", "test_data": "data/datasets/deepcrack_yolo/dataset.yaml", "ckpt": "runs/baselines/v8_crack500_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_crack500_baseline", "train": "Crack500", "test_data": "data/datasets/deepcrack_yolo/dataset.yaml", "ckpt": "runs/baselines/v11_crack500_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_crack500_full_kd", "train": "Crack500", "test_data": "data/datasets/deepcrack_yolo/dataset.yaml", "ckpt": "runs/crack_distill_full_kd_box_instance_seg_yolo11n-seg/weights/best.pt", "type": "Full KD"},

    # Train: DeepCrack -> Test: Crack500
    {"name": "v8_deepcrack_baseline", "train": "DeepCrack", "test_data": "data/datasets/crack500_yolo/dataset.yaml", "ckpt": "runs/baselines/v8_deepcrack_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_deepcrack_baseline", "train": "DeepCrack", "test_data": "data/datasets/crack500_yolo/dataset.yaml", "ckpt": "runs/baselines/v11_deepcrack_baseline/weights/best.pt", "type": "Baseline"},
    {"name": "v11_deepcrack_full_kd", "train": "DeepCrack", "test_data": "data/datasets/crack500_yolo/dataset.yaml", "ckpt": "runs/crack_distill_full_kd_box_instance_seg_deepcrack/weights/best.pt", "type": "Full KD"},
]

results = []
for item in eval_matrix:
    ckpt_path = Path(item["ckpt"])
    test_yaml = Path(item["test_data"])
    if ckpt_path.exists() and test_yaml.exists():
        print(f"Evaluating {item['name']} on {item['test_data']}...")
        model = YOLO(str(ckpt_path))
        metrics = model.val(data=str(test_yaml), split="test")
        results.append({
            "Experiment": item["name"],
            "Train Domain": item["train"],
            "Test Domain": "DeepCrack" if "deepcrack" in item["test_data"] else "Crack500",
            "Model Type": item["type"],
            "mAP50-seg": metrics.seg.map50,
            "mAP50-95-seg": metrics.seg.map,
            "mAP50-box": metrics.box.map50,
        })
    else:
        print(f"Skipping {item['name']}: checkpoint ({ckpt_path}) or test yaml ({test_yaml}) not found.")

df = pd.DataFrame(results)
print("\n" + "="*70)
print("🌐 CROSS-DATASET GENERALIZATION EVALUATION SUMMARY")
print("="*70)

if not df.empty:
    print(df.to_string(index=False))
else:
    print("Empty DataFrame (No checkpoints found to evaluate)")
    print("\n⚠️ INSTRUCTIONS TO RESOLVE:")
    print("   1. Click '+ Add Data' in your Kaggle Notebook sidebar.")
    print("   2. Select and add output datasets from your previous run notebooks (nb1, nb2, nb3).")
    print("   3. Re-run this notebook. Cell 5 will automatically detect, symlink, and evaluate all .pt files.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Evaluating v8_crack500_baseline on data/datasets/deepcrack_yolo/dataset.yaml...
Ultralytics 8.4.106 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,259 parameters, 0 gradients, 12.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2405.4±871.3 MB/s, size: 206.8 KB)
val: Scanning /kaggle/working/data/datasets/deepcrack_yolo/labels/test... 237 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 237/237 769.0it/s 0.3s
val: New cache created: /kaggle/working/data/datasets/deepcrack_yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R   